In [114]:
import pandas as pd

## Sankey diagram

In [115]:
africa = pd.read_csv("../resources/og/Africa_aggregated_data_up_to-2025-10-18.csv")
asiapacific = pd.read_csv("../resources/og/Asia-Pacific_aggregated_data_up_to-2025-10-11_0.csv")
europecentralasia = pd.read_csv("../resources/og/Europe-Central-Asia_aggregated_data_up_to-2025-10-11.csv")
latinamericacaribbean = pd.read_csv("../resources/og/Latin-America-the-Caribbean_aggregated_data_up_to-2025-10-18.csv")
middleeast = pd.read_csv("../resources/og/Middle-East_aggregated_data_up_to-2025-10-18.csv")
uscanada = pd.read_csv("../resources/og/US-and-Canada_aggregated_data_up_to-2025-10-11_0.csv")

sankey = pd.concat([africa, asiapacific, europecentralasia, latinamericacaribbean, middleeast, uscanada])

In [116]:
# Nodes
sankey["EVENT_NODE"] = "E:" + sankey["EVENT_TYPE"].astype(str)
sankey["SUB_NODE"] = "S:" + sankey["SUB_EVENT_TYPE"].astype(str)
sankey["DISORDER_NODE"] = "D:" + sankey["DISORDER_TYPE"].astype(str)


#Links
#EVENT → SUB_EVENT
event_to_sub = (
    sankey.groupby(["EVENT_NODE", "SUB_NODE"])
        .size()
        .reset_index(name="value")
)

#SUB_EVENT → DISORDER
sub_to_disorder = (
    sankey.groupby(["SUB_NODE", "DISORDER_NODE"])
        .size()
        .reset_index(name="value")
)

# Count each node total volume
event_counts = sankey["EVENT_NODE"].value_counts().reset_index()
event_counts.columns = ["name", "value"]

sub_counts = sankey["SUB_NODE"].value_counts().reset_index()
sub_counts.columns = ["name", "value"]

disorder_counts = sankey["DISORDER_NODE"].value_counts().reset_index()
disorder_counts.columns = ["name", "value"]

nodes = pd.concat([event_counts, sub_counts, disorder_counts], ignore_index=True)
nodes = nodes.groupby("name", as_index=False)["value"].sum()

#Map
node_to_id = {name: idx for idx, name in enumerate(nodes["name"])}

#Link names → IDs
event_to_sub["source"] = event_to_sub["EVENT_NODE"].map(node_to_id)
event_to_sub["target"] = event_to_sub["SUB_NODE"].map(node_to_id)

sub_to_disorder["source"] = sub_to_disorder["SUB_NODE"].map(node_to_id)
sub_to_disorder["target"] = sub_to_disorder["DISORDER_NODE"].map(node_to_id)

#Combine links
links = pd.concat([
    event_to_sub[["source", "target", "value"]],
    sub_to_disorder[["source", "target", "value"]]
], ignore_index=True)

#Create json
nodes.to_json("../resources/plots/sectionfive/sankey_nodes.json", orient="records")
links.to_json("../resources/plots/sectionfive/sankey_links.json", orient="records")

## Network